In [ ]:
import os
import json
import zipfile
import urllib.request

import numpy as np

RNG = np.random.default_rng(42)

TEXT8_URL     = "http://mattmahoney.net/dc/text8.zip"
TEXT8_PATH    = "text8"
TEXT8_ZIP     = "text8.zip"
HALUEVAL_URL  = "https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/qa_data.json"
HALUEVAL_PATH = "halueval.jsonl"
ANALOGY_PATH  = "analogies.txt"
ANALOGY_URL   = "https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt"

def load_text8(path=TEXT8_PATH):
    """
    Loads the Text8 corpus, downloading it automatically if not present.
    Source: http://mattmahoney.net/dc/text8.zip
    """
    if not os.path.exists(path):
        if not os.path.exists(TEXT8_ZIP):
            print("Downloading Text8...")
            urllib.request.urlretrieve(TEXT8_URL, TEXT8_ZIP)
        print("Extracting Text8...")
        with zipfile.ZipFile(TEXT8_ZIP) as zf:
            zf.extractall()
    with open(path) as f:
        return f.read().split()


def load_halueval(path=HALUEVAL_PATH):
    """
    Downloads HaluEval QA dataset if not present and converts to jsonl.
    Source: https://github.com/RUCAIBox/HaluEval
    """
    if not os.path.exists(path):
        print("Downloading HaluEval...")
        urllib.request.urlretrieve(HALUEVAL_URL, path)
        # Count lines to report sample count
        with open(path, encoding="utf-8") as f:
            n = sum(1 for _ in f)
        print(f"HaluEval saved to '{path}' ({n:,} samples).")
    return path

def load_analogy_dataset(path=ANALOGY_PATH):
    """
    Downloads the original Google word analogy benchmark if not present.
    Contains 19,544 analogies across semantic and syntactic categories.
    Source: https://github.com/tmikolov/word2vec
    """
    if not os.path.exists(path):
        print("Downloading Google analogy benchmark...")
        urllib.request.urlretrieve(ANALOGY_URL, path)
        with open(path, encoding="utf-8") as f:
            n = sum(1 for l in f if not l.startswith(":"))
        print(f"Analogy dataset saved to '{path}' ({n:,} analogies).")
    return path


def build_vocab(tokens, min_count=5):
    """
    Builds vocabulary without the Counter library.
    Keeps only words with frequency >= min_count.
    Sorts by frequency — most frequent words get lowest IDs.
    """
    freq = {}
    for t in tokens:
        freq[t] = freq.get(t, 0) + 1
    freq         = {w: c for w, c in freq.items() if c >= min_count}
    sorted_vocab = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    vocab        = {word: i for i, (word, _) in enumerate(sorted_vocab)}
    id2word      = {i: word for word, i in vocab.items()}
    return vocab, id2word, freq


def subsample(token_ids, freq, id2word, vocab_size, t=1e-5):
    """
    Mikolov subsampling of frequent words.
    P(keep) = sqrt(t / f(w))
    Frequent words (the, a, is) are discarded with high probability.
    """
    total  = sum(freq.values())
    f      = np.array([freq[id2word[i]] / total for i in range(vocab_size)])
    p_keep = np.sqrt(t / (f + 1e-12))
    keep   = p_keep[token_ids] > RNG.random(len(token_ids))
    return np.array(token_ids)[keep].tolist()


def make_noise_dist(freq, vocab, alpha=0.75):
    """
    Unigram^0.75 distribution for negative sampling.
    Exponent 0.75 reduces dominance of frequent words as negatives
    without fully equalising to a uniform distribution.
    """
    counts = np.array([freq[w] for w in vocab], dtype=np.float64)
    probs  = counts ** alpha
    return probs / probs.sum()


def skipgram_pairs_batch(token_ids, window, batch_size):
    """
    Generates (center, context) pairs with a dynamic window.
    c ~ Uniform[1, window] — closer words get a higher effective weight
    because they are included in more pairs than distant words.
    """
    centers, contexts = [], []
    for i, center in enumerate(token_ids): 
        c = RNG.integers(1, window + 1)
        lo, hi = max(0, i - c), min(len(token_ids), i + c + 1)
        
        for j in range(lo, hi):
            if i != j:
                centers.append(center)
                contexts.append(token_ids[j])
                
                if len(centers) == batch_size:
                
                    yield np.array(centers), np.array(contexts), i
                    centers, contexts = [], []
    if centers:
        yield np.array(centers), np.array(contexts), len(token_ids)

def sigmoid(x):
    """Numerically stable sigmoid."""
    return 1 / (1 + np.exp(-np.clip(x, -15, 15)))


def save_model(model, vocab, id2word, path="w2v_model"):
    os.makedirs(path, exist_ok=True)
    np.save(os.path.join(path, "W_in.npy"),  model.W_in)
    np.save(os.path.join(path, "W_out.npy"), model.W_out)
    with open(os.path.join(path, "vocab.json"), "w") as f:
        json.dump({
            "vocab"  : vocab,
            "id2word": {str(k): v for k, v in id2word.items()}
        }, f)
    print(f"Model saved to '{path}/'")


def load_model(path="w2v_model"):
    W_in  = np.load(os.path.join(path, "W_in.npy"))
    W_out = np.load(os.path.join(path, "W_out.npy"))
    with open(os.path.join(path, "vocab.json")) as f:
        data = json.load(f)
    vocab   = data["vocab"]
    id2word = {int(k): v for k, v in data["id2word"].items()}
    model   = Word2Vec(vocab_size=W_in.shape[0], dim=W_in.shape[1])
    model.W_in, model.W_out = W_in, W_out
    print(f"Model loaded from '{path}/'")
    return model, vocab, id2word

class Word2Vec:
    """
    Two embedding matrices:
      W_in  (V, D) — center-word vectors  (used as final word vectors)
      W_out (V, D) — context-word vectors (initialised to zeros)
    """

    def __init__(self, vocab_size, dim=100):
        self.D     = dim
        self.W_in  = (RNG.random((vocab_size, dim)) - 0.5) / dim
        self.W_out = (RNG.random((vocab_size, dim)) - 0.5) / dim

    # ── Forward pass + loss ──────────────────────────────────
    def forward_loss_batch(self, centers, contexts, negatives):
        """
        SGNS loss for a batch of B pairs:

          L = -mean_b [ log σ(u_pos·v_c) + Σ_k log σ(-u_neg_k·v_c) ]

        Gradients derived via chain rule through sigmoid:
          ∂L/∂v_c     = (σ(s_pos)-1)·u_pos + Σ_k σ(s_neg_k)·u_neg_k
          ∂L/∂u_pos   = (σ(s_pos)-1)·v_c
          ∂L/∂u_neg_k = σ(s_neg_k)·v_c
        """
        vc    = self.W_in[centers]       # (B, D)
        u_pos = self.W_out[contexts]     # (B, D)
        u_neg = self.W_out[negatives]    # (B, K, D)

        s_pos = np.sum(vc * u_pos, axis=1)             # (B,)
        s_neg = np.einsum('bd,bkd->bk', vc, u_neg)     # (B, K)

        sig_pos = sigmoid(s_pos)
        sig_neg = sigmoid(s_neg)

        eps  = 1e-7
        loss = -np.mean(
            np.log(sig_pos + eps) + np.sum(np.log(1 - sig_neg + eps), axis=1)
        )

        B       = len(centers)
        err_pos = (sig_pos - 1.0) 
        err_neg = sig_neg 

        grad_vc    = err_pos[:, None] * u_pos + np.einsum('bk,bkd->bd', err_neg, u_neg)
        grad_u_pos = err_pos[:, None] * vc
        grad_u_neg = np.einsum('bk,bd->bkd', err_neg, vc)

        return loss, grad_vc, grad_u_pos, grad_u_neg

    # ── SGD update ───────────────────────────────────────────
    def update_batch(self, centers, contexts, negatives, g_vc, g_u_pos, g_u_neg, lr):
        """
        np.add.at accumulates gradients for repeated indices within a batch.
        Plain indexing W_in[centers] -= grad would overwrite duplicates.
        """
        np.add.at(self.W_in,  centers,          -lr * g_vc)
        np.add.at(self.W_out, contexts,          -lr * g_u_pos)
        np.add.at(self.W_out, negatives.ravel(), -lr * g_u_neg.reshape(-1, self.D))

    # ── Vector evaluation ────────────────────────────────────
    def most_similar(self, word, vocab, id2word, top_n=8):
        """Cosine similarity — top N nearest neighbours."""
        if word not in vocab:
            print(f"  '{word}' not in vocabulary.")
            return
        idx   = vocab[word]
        vec   = self.W_in[idx]
        norms = np.linalg.norm(self.W_in, axis=1)
        sims  = self.W_in @ vec / (norms * np.linalg.norm(vec) + 1e-9)
        sims[idx] = -1
        top = np.argsort(sims)[::-1][:top_n]
        print(f"  Nearest neighbours of '{word}':")
        for i in top:
            print(f"    {id2word[i]:<20s}  {sims[i]:.4f}")

    def analogy(self, a, b, c, vocab, id2word, top_n=1):
        """
        Vector analogy: a - b + c = ?
        Example: king - man + woman = queen
        Tests the linear structure of the embedding space.
        Returns a list of top_n predictions.
        """
        for w in [a, b, c]:
            if w not in vocab:
                return None
        target = self.W_in[vocab[a]] - self.W_in[vocab[b]] + self.W_in[vocab[c]]
        norms  = np.linalg.norm(self.W_in, axis=1)
        sims   = self.W_in @ target / (norms * np.linalg.norm(target) + 1e-9)
        for w in [a, b, c]:
            sims[vocab[w]] = -1
        top = np.argsort(sims)[::-1][:top_n]
        return [id2word[i] for i in top]

    # ── SGI methods ──────────────────────────────────────────
    def get_vector(self, text, vocab):
        """
        Sentence vector = mean of W2V word vectors in the text.
        Limitation: loses word order and syntax — see SGI results.
        """
        tokens = text.lower().split()
        idxs   = [vocab[t] for t in tokens if t in vocab]
        return np.mean(self.W_in[idxs], axis=0) if idxs else np.zeros(self.D)

    def angular_distance(self, v1, v2):
        """
        Geodesic distance on S^{d-1}: θ(a,b) = arccos(â·b̂)
        Unlike cosine similarity, satisfies the triangle inequality —
        which is required for the SGI theoretical bounds (Marin 2025).
        """
        v1_u = v1 / (np.linalg.norm(v1) + 1e-9)
        v2_u = v2 / (np.linalg.norm(v2) + 1e-9)
        return np.arccos(np.clip(np.dot(v1_u, v2_u), -1, 1))

    def compute_sgi(self, q, c, r, vocab):
        """
        SGI = θ(r,q) / θ(r,c)

        SGI > 1 : response is angularly farther from the question
                  than from the context → grounded, valid answer
        SGI < 1 : response stays close to the question →
                  semantic laziness → hallucination signal
        """
        qv = self.get_vector(q, vocab)
        cv = self.get_vector(c, vocab)
        rv = self.get_vector(r, vocab)
        return self.angular_distance(rv, qv) / (self.angular_distance(rv, cv) + 1e-8)


def train(token_ids, model, noise_probs, epochs=3, batch_size=512,
          lr_start=0.025, lr_min=0.0001, report_every=100000):
    
    N = len(token_ids)
    total_steps = N * epochs 

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        n_batches = 0
      
        for centers, contexts, word_idx in skipgram_pairs_batch(token_ids, 5, batch_size):
            
            current_total_words = ((epoch - 1) * N) + word_idx
            
            lr = max(lr_start * (1 - current_total_words / total_steps), lr_min)

            B = len(centers)
            negs = RNG.choice(len(noise_probs), size=(B, 5), p=noise_probs)
            loss, g_vc, g_u_pos, g_u_neg = model.forward_loss_batch(centers, contexts, negs)
            model.update_batch(centers, contexts, negs, g_vc, g_u_pos, g_u_neg, lr)

            epoch_loss += loss
            n_batches += 1
            
            if (n_batches * batch_size) % report_every < batch_size:
                print(f"Epoch {epoch} | Progress: {word_idx/N*100:.1f}% | LR: {lr:.5f} | Loss: {epoch_loss/n_batches:.4f}")

    return model


def evaluate_analogies(model, vocab, id2word, path=ANALOGY_PATH):
    """
    Evaluates analogy accuracy on the local analogy file.

    For each line 'a b c d':
      Predicts top-1 word for  a - b + c
      Correct if prediction == d

    Accuracy = correct / total (skipped = words not in vocabulary)
    """
    if not os.path.exists(path):
        print(f"Analogy file not found at '{path}'.")
        return

    correct = 0
    total   = 0
    skipped = 0

    print("\n" + "=" * 55)
    print("ANALOGY ACCURACY EVALUATION")
    print("=" * 55)

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(":"):
                continue

            parts = line.split()
            if len(parts) != 4:
                continue

            a, b, c, expected = parts

            if any(w not in vocab for w in [a, b, c, expected]):
                skipped += 1
                continue

            predictions = model.analogy(a, b, c, vocab, id2word, top_n=1)
            if predictions is None:
                skipped += 1
                continue

            if predictions[0] == expected:
                correct += 1
            total += 1

    if total == 0:
        print("  No valid analogies found.")
        return

    print(f"\n  Accuracy : {correct}/{total} = {correct/total*100:.1f}%")
    print(f"  Skipped  : {skipped} (words not in vocabulary)")

def evaluate_sgi(model, vocab, path=HALUEVAL_PATH, max_samples=500):
    """
    Evaluates SGI hallucination detection on the local HaluEval file.

    HaluEval format (jsonl — one JSON object per line):
    {
      "question"            : "...",
      "knowledge"           : "...",   <- retrieved context
      "right_answer"        : "...",
      "hallucinated_answer" : "..."
    }

    Metric: Accuracy = % where SGI(valid) > SGI(hallucinated)

    Expected results:
      Sentence transformers (paper) : ~80% accuracy
      Word2Vec mean pooling (ours)  : ~55-65% accuracy
    """
    if not os.path.exists(path):
        print(f"HaluEval file not found at '{path}'.\n"
              f"Download from https://github.com/RUCAIBox/HaluEval")
        return

    sgi_valid_list  = []
    sgi_halluc_list = []
    n_skipped       = 0

    with open(path, encoding="utf-8") as f:
        for line_num, line in enumerate(f):
            if line_num >= max_samples:
                break
            try:
                s = json.loads(line)
            except json.JSONDecodeError:
                continue

            qv  = model.get_vector(s.get("question", ""),            vocab)
            cv  = model.get_vector(s.get("knowledge", ""),           vocab)
            rvv = model.get_vector(s.get("right_answer", ""),        vocab)
            rhv = model.get_vector(s.get("hallucinated_answer", ""), vocab)

            if any(np.all(v == 0) for v in [qv, cv, rvv, rhv]):
                n_skipped += 1
                continue

            sgi_valid_list.append(
                model.compute_sgi(s["question"], s["knowledge"],
                                  s["right_answer"], vocab))
            sgi_halluc_list.append(
                model.compute_sgi(s["question"], s["knowledge"],
                                  s["hallucinated_answer"], vocab))

    if not sgi_valid_list:
        print("  No valid samples for evaluation.")
        return

    sgi_v = np.array(sgi_valid_list)
    sgi_h = np.array(sgi_halluc_list)
    acc   = np.mean(sgi_v > sgi_h)

    print("\n" + "=" * 55)
    print("SGI HALLUCINATION EVALUATION (W2V vectors)")
    print("=" * 55)
    print(f"  Samples evaluated : {len(sgi_v)}  (skipped: {n_skipped})")
    print(f"  SGI valid         : {np.mean(sgi_v):.3f} ± {np.std(sgi_v):.3f}")
    print(f"  SGI hallucinated  : {np.mean(sgi_h):.3f} ± {np.std(sgi_h):.3f}")
    print(f"  Accuracy          : {acc*100:.1f}%")


if __name__ == "__main__":
    DIM        = 100
    EPOCHS     = 3
    BATCH_SIZE = 512
    MAX_TOKENS = None  # use all tokens

    # ── Generate / download datasets ────────────────────────
    load_analogy_dataset()
    load_halueval()

    # ── Load Text8 and train ─────────────────────────────────
    print("=" * 55)
    print("Word2Vec SGNS")
    print("=" * 55)

    tokens = load_text8()
    print(f"Corpus     : {len(tokens):,} tokens")

    vocab, id2word, freq = build_vocab(tokens, min_count=5)
    print(f"Vocabulary : {len(vocab):,} words")

    token_ids   = [vocab[t] for t in tokens if t in vocab]
    token_ids   = subsample(token_ids, freq, id2word, len(vocab))

    print(f"After subsampling: {len(token_ids):,} tokens")

    noise_probs = make_noise_dist(freq, vocab)

    model = Word2Vec(vocab_size=len(vocab), dim=DIM)
    print(f"\nTraining ({EPOCHS} epochs, batch={BATCH_SIZE})...\n")
    train(token_ids, model, noise_probs, epochs=EPOCHS, batch_size=BATCH_SIZE)
    save_model(model, vocab, id2word)

    # ── W2V vector evaluation ────────────────────────────────
    print("\n" + "=" * 55)
    print("W2V VECTOR EVALUATION — most similar")
    print("=" * 55)

    # examples
    for word in ["king", "paris", "computer", "music"]:
        model.most_similar(word, vocab, id2word)
        print()

    # ── Analogy accuracy ─────────────────────────────────────
    evaluate_analogies(model, vocab, id2word)

    # ── SGI hallucination evaluation ─────────────────────────
    evaluate_sgi(model, vocab, path=HALUEVAL_PATH, max_samples=500)

Word2Vec SGNS
Corpus     : 17,005,207 tokens
Vocabulary : 71,290 words
After subsampling: 4,668,656 tokens

Training (3 epochs, batch=512)...

Epoch 1 | Progress: 0.4% | LR: 0.02497 | Loss: 4.1503
Epoch 1 | Progress: 0.7% | LR: 0.02494 | Loss: 4.0799
Epoch 1 | Progress: 1.1% | LR: 0.02491 | Loss: 3.9893
Epoch 1 | Progress: 1.4% | LR: 0.02488 | Loss: 3.9010
Epoch 1 | Progress: 1.8% | LR: 0.02485 | Loss: 3.8246
Epoch 1 | Progress: 2.1% | LR: 0.02482 | Loss: 3.7550
Epoch 1 | Progress: 2.5% | LR: 0.02479 | Loss: 3.6970
Epoch 1 | Progress: 2.9% | LR: 0.02476 | Loss: 3.6387
Epoch 1 | Progress: 3.2% | LR: 0.02473 | Loss: 3.5936
Epoch 1 | Progress: 3.6% | LR: 0.02470 | Loss: 3.5511
Epoch 1 | Progress: 3.9% | LR: 0.02467 | Loss: 3.5146
Epoch 1 | Progress: 4.3% | LR: 0.02464 | Loss: 3.4794
Epoch 1 | Progress: 4.6% | LR: 0.02461 | Loss: 3.4434
Epoch 1 | Progress: 5.0% | LR: 0.02458 | Loss: 3.4120
Epoch 1 | Progress: 5.4% | LR: 0.02455 | Loss: 3.3844
Epoch 1 | Progress: 5.7% | LR: 0.02452 | Loss: 